## Status (created 2026-08-04) -- READ BEFORE RUNNING

**This one trains.** GPU runtime required. Budget ~4 arms x 3 seeds = 12 Stage-2 runs on
pancreas.

Reviewer nG29 (round 3): *"variance in the ablations is reported across batches rather
than seeds, which makes deriving conclusions from these results difficult. [...]
Reporting variance over seeds and for all datasets would make these results clearer."*

**What a seed run answers that a batch run cannot.** Per-batch spread says how much
batches differ from each other. It cannot say whether an effect would survive a
different random initialisation. Those are different questions, and only re-training
answers the second one. (The first is answered without re-training by
`ablation_paired_significance.ipynb` -- run that one too; the two are complements, not
alternatives.)

**The arm that matters most is the one the reviewer did not ask for.** He named
`- nassoc` and `stop-gradient`. This notebook also runs `- community loss`, and that is
the point of the whole exercise: it is the largest effect in the ablation
(pancreas modularity .615 -> .449). Once seed spread is known, every arm gets a scale --
*"the community-loss drop is N times the seed noise"*. Without a large-effect arm in the
same run there is no yardstick and the seed numbers say nothing.

### Two deliberate choices

**Stage 1 is shared across seeds; only Stage 2 is re-seeded.** `seed` is not in
`PRETRAIN_PARAM_KEYS` (`adaptive_trainer.py:65`), so every arm resolves to the same
Stage-1 checkpoint. That is the correct scope here -- the ablation is about Stage-2
components, and every arm in the original ablation also shared one Stage-1. It is also
much faster.

**`cvae_epochs` stays at 50 -- do NOT lower it.** It is part of the Stage-1 checkpoint
name, so lowering it forces a *fresh Stage-1 pretrain for every run*, making the notebook
slower, not faster, and breaking comparability with the reported table. Only
`train_epochs` (Stage 2) is reduced.

**These numbers will not match the paper's table**, because Stage 2 is shortened. That is
fine and it is the point: every arm here runs at the *same* reduced setting, so
`full - ablated` and the seed spread are measured on the same footing and can be compared
to each other. Report this as a stability check, labelled as such -- not as a replacement
for the main ablation table.

Every finished (seed, arm) is written to `OUT_DIR` immediately, so a Colab disconnect
costs at most one run.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 158.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 62.2 MB/s eta 0:00:00
   ━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 131.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 whi

In [ ]:
# Restart the runtime after the install cell above before running anything below --
# numpy/scipy/anndata are C-extension linked and an in-process upgrade won't take
# effect on already-imported modules.

In [1]:
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'torch': 'torch',
}
_failed = []
for pkg, imp in _checks.items():
    try:
        m = __import__(imp)
        print(f'  OK   {pkg:14s} {getattr(m, "__version__", "?")}')
    except Exception as e:
        _failed.append(pkg)
        print(f'  FAIL {pkg:14s} {type(e).__name__}: {e}')
print('\nall good' if not _failed else f'\nfailed: {_failed}')

import torch
print('cuda available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

  OK   numpy          2.2.6
  OK   scipy          1.13.1


/tmp/ipykernel_2064/654138299.py:9: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f'  OK   {pkg:14s} {getattr(m, "__version__", "?")}')


  OK   anndata        0.13.2


/tmp/ipykernel_2064/654138299.py:9: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f'  OK   {pkg:14s} {getattr(m, "__version__", "?")}')


  OK   scanpy         1.12.3


  FAIL scarches       ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools     1.5.0.post1
  OK   torch          2.11.0+cu128

failed: ['scarches']
cuda available: True | NVIDIA A100-SXM4-40GB


In [2]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [3]:
import os, json, time, traceback
import numpy as np
import pandas as pd

from interpretable_ssl.experiments.tasks import (
    run_ablation_variant, LAMBDA_PROTO_UMAP_PRECON,
)
from interpretable_ssl.configs.paths import get_dataset_model_dir

print('ready')

ready


## Config

The overrides below are read off `LAMBDA_PROTO_UMAP_PRECON` (`tasks.py:35`), the exact
Stage-2 objective the paper's runs use:

| arm | override | what it removes |
|---|---|---|
| `full` | -- | nothing (reference) |
| `no_community` | `lambda_umap=0` | the affinity/community loss -- **the yardstick arm** |
| `no_nassoc` | `lambda_nassoc=0` | nassoc (reviewer named this one) |
| `stopgrad_off` | `proto_recon_stopgrad=0` | the stop-gradient (reviewer named this one) |

`proto_recon_stopgrad` is read via `getattr(self, 'proto_recon_stopgrad', True)`
(`scproto.py:1460`) and reaches the trainer because `run_mc_task` forwards every
`lambda_config` key to `get_trainer` (`tasks.py:249`).

Seed 31 is first on purpose: it is the codebase default (`configs/defaults.py:47`), so
that arm is the closest thing to a re-run of the original ablation and acts as a sanity
check on the whole setup.

In [4]:
DS = 'pancreas'          # one dataset first -- extend below only if time allows
SEEDS = [31, 1, 2]       # 31 = codebase default, so it doubles as a sanity check

ARMS = {
    'full':         {},
    'no_community': {'lambda_umap': 0},
    'no_nassoc':    {'lambda_nassoc': 0},
    'stopgrad_off': {'proto_recon_stopgrad': 0},
}
ARM_DISPLAY = {
    'full':         'Full model',
    'no_community': '- community loss',
    'no_nassoc':    '- nassoc',
    'stopgrad_off': 'Stop-grad off',
}

# cvae_epochs MUST stay 50 -- it names the shared Stage-1 checkpoint (see intro).
# train_epochs is the only thing reduced.
COMMON = dict(
    cvae_epochs=50,
    train_epochs=20,
    eval_freq=1,
    patience=5,
    batch_size=1024,
)

OUT_DIR = '/content/drive/MyDrive/rebuttal_results/ablation_seeds/'
os.makedirs(OUT_DIR, exist_ok=True)

SKIP_IF_DONE = True   # a (ds, seed, arm) whose record already exists is not re-run

print(f'{len(SEEDS)} seeds x {len(ARMS)} arms = {len(SEEDS) * len(ARMS)} runs on {DS}')
print('OUT_DIR:', OUT_DIR)

3 seeds x 4 arms = 12 runs on pancreas
OUT_DIR: /content/drive/MyDrive/rebuttal_results/ablation_seeds/


## Runner

Two layers of resume, because a Colab session dying mid-sweep is the normal case:

1. **Record level** -- if `OUT_DIR/{ds}__s{seed}__{arm}.json` exists, the run is skipped
   entirely (no reload, no re-eval).
2. **Checkpoint level** -- `run_ablation_variant(load_umap=None)` auto-detects a saved
   Stage-2 checkpoint for that exact `experiment_name` and reloads instead of retraining
   (`tasks.py:434`). So even a record deleted by hand costs only re-evaluation.

`experiment_prefix=f'seedabl_s{seed}'` is what keeps the seeds apart on disk. It has to
be the prefix, because `seed` does not appear anywhere in the model-name builder -- two
seeds run under the same prefix would resolve to the same folder and silently overwrite
each other.

The per-batch modularity vector is copied into the record as well as the scalar mean, so
the paired-test notebook can be pointed at these runs too without re-reading the model
dir.

In [5]:
def record_path(ds, seed, arm):
    return os.path.join(OUT_DIR, f'{ds}__s{seed}__{arm}.json')


def _per_batch_modularity(ds, seed, arm):
    """Copy modularity_per_batch.csv out of this run's folder into the record."""
    base = get_dataset_model_dir(ds)
    prefix = f'seedabl_s{seed}_{arm}_'
    cands = [e for e in sorted(os.listdir(base)) if e.startswith(prefix)] if os.path.isdir(base) else []
    for e in reversed(cands):
        p = os.path.join(base, e, 'modularity_per_batch.csv')
        if os.path.exists(p):
            df = pd.read_csv(p, index_col=0)
            num = df.select_dtypes(include=[np.number])
            if num.shape[1]:
                s = num.iloc[:, 0].astype(float)
                return {'batches': [str(i) for i in s.index], 'values': [float(v) for v in s.values]}
    return None


def run_one(ds, seed, arm, force=False):
    path = record_path(ds, seed, arm)
    if SKIP_IF_DONE and not force and os.path.exists(path):
        print(f'  [{ds} s{seed} {arm}] record exists -- skipped')
        with open(path) as f:
            return json.load(f)

    overrides = dict(ARMS[arm])
    overrides['seed'] = seed      # -> get_trainer -> fix_random_seeds(self.seed)

    t0 = time.time()
    try:
        _, res, _ = run_ablation_variant(
            ds, arm, overrides,
            base_lambda_config=LAMBDA_PROTO_UMAP_PRECON,
            common_kwargs=COMMON,
            variant_display_names=ARM_DISPLAY,
            load_umap=None,                       # auto: reload checkpoint if present
            experiment_prefix=f'seedabl_s{seed}',  # keeps seeds in separate folders
        )
    except Exception:
        print(f'  [{ds} s{seed} {arm}] FAILED after {time.time() - t0:.0f}s')
        traceback.print_exc()
        return None

    rec = {'dataset': ds, 'seed': seed, 'arm': arm,
           'elapsed_s': round(time.time() - t0, 1),
           'metrics': {k: v for k, v in res.items() if isinstance(v, (int, float, type(None)))},
           'modularity_per_batch': _per_batch_modularity(ds, seed, arm)}

    with open(path, 'w') as f:
        json.dump(rec, f, indent=2)
    print(f'  [{ds} s{seed} {arm}] done in {rec["elapsed_s"]}s -> {path}')
    return rec

print('runner ready')

runner ready


## Run: Pancreas

Arm-major, not seed-major: all seeds of `full` finish before `no_community` starts. If
the session dies halfway you have complete seed spread for the arms that ran, instead of
one incomplete seed of everything.

In [6]:
records = []
for arm in ARMS:
    for seed in SEEDS:
        r = run_one(DS, seed, arm)
        if r:
            records.append(r)

print(f'\n{len(records)}/{len(ARMS) * len(SEEDS)} runs available')


=== [pancreas] ablation variant: full (Full model) ===  [training fresh]


 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
seedabl_s31_full_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
[waypoint init] N=16382  K=220  n_eigs=10  nnz=1155146  nnz/row=70.5  w[min/mean/max]=3.203e-02/3.392e-01/9.688e-01  deg[mi

waypoint MaxMin: 100%|██████████| 219/219 [00:00<00:00, 2677.41proto/s]

[waypoint init] selected 220 seed cells


...

  0%|          | 0/16 [00:00<?, ?it/s]

Saved metacells (220 prototypes) to /content/drive/MyDrive/models//pancreas/seedabl_s2_stopgrad_off_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/metacells.h5ad


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//pancreas/seedabl_s2_stopgrad_off_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
  [pancreas s2 stopgrad_off] done in 644.8s -> /content/drive/MyDrive/rebuttal_results/ablation_seeds/pancreas__s2__stopgrad_off.json

12/12 runs available


## Analysis 1 -- spread across seeds, per arm

This is the table nG29 asked for: `mean +- std` where the `+-` is **over seeds**.

In [7]:
def load_records(ds):
    out = []
    for f in sorted(os.listdir(OUT_DIR)):
        if f.startswith(f'{ds}__') and f.endswith('.json'):
            with open(os.path.join(OUT_DIR, f)) as fh:
                out.append(json.load(fh))
    return out


METRICS = ['modularity', 'purity', 'batch_entropy', 'coverage',
           'batch_rare_homogeneity_mean', 'batch_rare_coverage_mean',
           'batch_rare_f1_macro_mean', 'active_prototype_count']

recs = load_records(DS)
rows = []
for r in recs:
    row = {'arm': r['arm'], 'seed': r['seed']}
    row.update({m: r['metrics'].get(m) for m in METRICS})
    rows.append(row)

df_raw = pd.DataFrame(rows).sort_values(['arm', 'seed'])
present = [m for m in METRICS if m in df_raw.columns and df_raw[m].notna().any()]
missing = [m for m in METRICS if m not in present]
if missing:
    print(f'not produced by this run (name differs or metric skipped): {missing}')

print(f'\nraw per-seed values ({len(df_raw)} runs)')
display(df_raw[['arm', 'seed'] + present])

df_seed = df_raw.groupby('arm')[present].agg(['mean', 'std', 'count'])
print('\nmean +- std ACROSS SEEDS')
display(df_seed)

df_raw.to_csv(os.path.join(OUT_DIR, f'{DS}_per_seed_raw.csv'), index=False)
df_seed.to_csv(os.path.join(OUT_DIR, f'{DS}_per_seed_summary.csv'))
print('saved ->', OUT_DIR)

not produced by this run (name differs or metric skipped): ['batch_rare_homogeneity_mean', 'batch_rare_coverage_mean', 'batch_rare_f1_macro_mean', 'active_prototype_count']

raw per-seed values (12 runs)


,arm,seed,modularity,purity,batch_entropy,coverage
0,full,1,0.701069,0.922393,0.445614,1.000000
4,full,2,0.697862,0.914601,0.376770,0.928571
8,full,31,0.689617,0.911317,0.434529,1.000000
1,no_community,1,0.509445,0.955964,0.468540,0.785714
5,no_community,2,0.497727,0.956642,0.486260,0.785714
9,no_community,31,0.488888,0.954927,0.478449,0.857143
2,no_nassoc,1,0.699537,0.906820,0.378727,0.928571
6,no_nassoc,2,0.690684,0.921303,0.379863,0.928571
10,no_nassoc,31,0.704029,0.872092,0.502007,0.928571
3,stopgrad_off,1,0.626371,0.902231,0.553437,0.928571



mean +- std ACROSS SEEDS


modularity                    purity                  \
                   mean       std count      mean       std count   
arm                                                                 
full           0.696183  0.005908     3  0.916104  0.005689     3   
no_community   0.498687  0.010312     3  0.955844  0.000863     3   
no_nassoc      0.698084  0.006790     3  0.900072  0.025290     3   
stopgrad_off   0.626395  0.001902     3  0.905739  0.004855     3   

             batch_entropy                  coverage                  
                      mean       std count      mean       std count  
arm                                                                   
full              0.418971  0.036965     3  0.976190  0.041239     3  
no_community      0.477750  0.008881     3  0.809524  0.041239     3  
no_nassoc         0.420199  0.070850     3  0.928571  0.000000     3  
stopgrad_off      0.578395  0.021654     3  0.928571  0.000000     3

saved -> /content/drive/MyDrive/rebuttal_results/ablation_seeds/


## Analysis 2 -- effect size against seed noise

The number to quote. For each arm and metric:

```
effect      = mean_over_seeds(full) - mean_over_seeds(arm)
seed_noise  = std_over_seeds(full)
ratio       = |effect| / seed_noise
```

`ratio` says how many seed-standard-deviations the ablation moved the metric. A ratio
near 1 means the arm is indistinguishable from re-running the same model with a different
initialisation -- and if that is what `stopgrad_off` shows, the honest move is to drop the
empirical claim and justify the stop-gradient by design instead.

`- community loss` is in here as the calibration point: whatever ratio it earns is the
scale against which every other arm should be read.

In [8]:
def effect_vs_seed_noise(df_raw, metrics, ref_arm='full'):
    if ref_arm not in set(df_raw['arm']):
        print(f'no {ref_arm!r} arm -- cannot compute effects')
        return pd.DataFrame()
    g = df_raw.groupby('arm')
    rows = []
    for m in metrics:
        if m not in df_raw.columns:
            continue
        ref_mean, ref_std = g[m].mean().get(ref_arm), g[m].std().get(ref_arm)
        n_ref = int(g[m].count().get(ref_arm, 0))
        for arm in df_raw['arm'].unique():
            if arm == ref_arm:
                continue
            am, asd = g[m].mean().get(arm), g[m].std().get(arm)
            if pd.isna(ref_mean) or pd.isna(am):
                continue
            eff = ref_mean - am
            pooled = np.sqrt(np.nanmean([ref_std ** 2, asd ** 2])) if pd.notna(ref_std) and pd.notna(asd) else np.nan
            rows.append({
                'metric': m, 'arm': arm,
                'full_mean': ref_mean, 'full_seed_std': ref_std,
                'arm_mean': am, 'arm_seed_std': asd,
                'effect': eff,
                'ratio_vs_full_std': abs(eff) / ref_std if pd.notna(ref_std) and ref_std > 0 else np.nan,
                'ratio_vs_pooled_std': abs(eff) / pooled if pd.notna(pooled) and pooled > 0 else np.nan,
                'n_seeds': min(n_ref, int(g[m].count().get(arm, 0))),
            })
    return pd.DataFrame(rows)


df_eff = effect_vs_seed_noise(df_raw, present)
if not df_eff.empty:
    df_eff = df_eff.sort_values(['metric', 'ratio_vs_full_std'], ascending=[True, False])
    display(df_eff.round(4))
    df_eff.to_csv(os.path.join(OUT_DIR, f'{DS}_effect_vs_seed_noise.csv'), index=False)
    print('saved ->', os.path.join(OUT_DIR, f'{DS}_effect_vs_seed_noise.csv'))

,metric,arm,full_mean,full_seed_std,arm_mean,arm_seed_std,effect,ratio_vs_full_std,ratio_vs_pooled_std,n_seeds
8,batch_entropy,stopgrad_off,0.4190,0.0370,0.5784,0.0217,-0.1594,4.3128,5.2627,3
6,batch_entropy,no_community,0.4190,0.0370,0.4777,0.0089,-0.0588,1.5901,2.1865,3
7,batch_entropy,no_nassoc,0.4190,0.0370,0.4202,0.0709,-0.0012,0.0332,0.0217,3
9,coverage,no_community,0.9762,0.0412,0.8095,0.0412,0.1667,4.0415,4.0415,3
10,coverage,no_nassoc,0.9762,0.0412,0.9286,0.0000,0.0476,1.1547,1.6330,3
11,coverage,stopgrad_off,0.9762,0.0412,0.9286,0.0000,0.0476,1.1547,1.6330,3
0,modularity,no_community,0.6962,0.0059,0.4987,0.0103,0.1975,33.4301,23.5021,3
2,modularity,stopgrad_off,0.6962,0.0059,0.6264,0.0019,0.0698,11.8129,15.9019,3
1,modularity,no_nassoc,0.6962,0.0059,0.6981,0.0068,-0.0019,0.3217,0.2986,3
3,purity,no_community,0.9161,0.0057,0.9558,0.0009,-0.0397,6.9856,9.7673,3


saved -> /content/drive/MyDrive/rebuttal_results/ablation_seeds/pancreas_effect_vs_seed_noise.csv


In [9]:
# Headline sentence, ready to paste.
if not df_eff.empty and 'modularity' in set(df_eff['metric']):
    d = df_eff[df_eff['metric'] == 'modularity'].set_index('arm')
    noise = d['full_seed_std'].iloc[0]
    print(f'Modularity, {DS}, across {int(d["n_seeds"].max())} seeds')
    print(f'  seed-to-seed std of the full model: {noise:.4f}\n')
    for arm in d.index:
        r = d.loc[arm]
        print(f'  {ARM_DISPLAY.get(arm, arm):18s} effect {r["effect"]:+.4f}  '
              f'= {r["ratio_vs_full_std"]:.1f}x the seed noise')
    print('\nRead as: an arm whose ratio is around 1 is not separable from re-running '
          'the same model with a different seed.')

Modularity, pancreas, across 3 seeds
  seed-to-seed std of the full model: 0.0059

  - community loss   effect +0.1975  = 33.4x the seed noise
  Stop-grad off      effect +0.0698  = 11.8x the seed noise
  - nassoc           effect -0.0019  = 0.3x the seed noise

Read as: an arm whose ratio is around 1 is not separable from re-running the same model with a different seed.


## Analysis 3 -- pooling seeds into the paired batch test

Seeds and batches are not competing evidence -- they stack. Pooling every (seed, batch)
pair for an arm against the same pairs for the full model gives `n_seeds x n_batches`
paired observations instead of `n_batches`, which is the strongest version of the
argument: the difference holds within each batch, *and* it holds across initialisations.

Pairing is on `(seed, batch)`, so a batch is only ever compared against itself under the
same seed.

In [10]:
from scipy.stats import wilcoxon

def pooled_paired_modularity(ds, ref_arm='full'):
    recs = load_records(ds)
    by = {}
    for r in recs:
        pb = r.get('modularity_per_batch')
        if pb:
            by[(r['arm'], r['seed'])] = pd.Series(pb['values'], index=pb['batches'], dtype=float)
    if not by:
        print('no per-batch modularity stored in the records yet')
        return pd.DataFrame()

    arms = sorted({a for a, _ in by})
    seeds = sorted({s for _, s in by})
    rows = []
    for arm in arms:
        if arm == ref_arm:
            continue
        ref_v, arm_v, keys = [], [], []
        for s in seeds:
            a, b = by.get((ref_arm, s)), by.get((arm, s))
            if a is None or b is None:
                continue
            a2, b2 = a.align(b, join='inner')
            ref_v += list(a2.values); arm_v += list(b2.values)
            keys += [(s, i) for i in a2.index]
        if not ref_v:
            continue
        ref_v, arm_v = np.array(ref_v), np.array(arm_v)
        diff = ref_v - arm_v
        row = {'arm': arm, 'n_pairs': len(diff), 'n_seeds': len({k[0] for k in keys}),
               'full_mean': ref_v.mean(), 'arm_mean': arm_v.mean(),
               'mean_diff': diff.mean(), 'n_drop': int((diff > 0).sum())}
        if np.any(diff != 0):
            row['p_two_sided'] = float(wilcoxon(ref_v, arm_v, alternative='two-sided')[1])
        rows.append(row)
    return pd.DataFrame(rows)


df_pooled = pooled_paired_modularity(DS)
if not df_pooled.empty:
    _show = df_pooled.round(4)
    if 'p_two_sided' in _show:   # can go far below 1e-4 once seeds are pooled --
        _show['p_two_sided'] = df_pooled['p_two_sided'].map('{:.3g}'.format)  # never print "0.0"
    display(_show)
    df_pooled.to_csv(os.path.join(OUT_DIR, f'{DS}_pooled_paired_modularity.csv'), index=False)
    print('saved ->', os.path.join(OUT_DIR, f'{DS}_pooled_paired_modularity.csv'))

,arm,n_pairs,n_seeds,full_mean,arm_mean,mean_diff,n_drop,p_two_sided
0,no_community,27,3,0.6171,0.4505,0.1666,26,4.47e-08
1,no_nassoc,27,3,0.6171,0.6429,-0.0259,4,8.37e-05
2,stopgrad_off,27,3,0.6171,0.5620,0.0550,26,2.98e-08


saved -> /content/drive/MyDrive/rebuttal_results/ablation_seeds/pancreas_pooled_paired_modularity.csv


## Extend to Lung / Immune -- only if pancreas is finished and time allows

Run these one at a time and check the pancreas result first. If the session runs out,
having pancreas complete is worth more than three datasets half-done -- pancreas is the
dataset the reviewer quoted.

In [11]:
# DS = 'lung'
# records = [run_one(DS, s, a) for a in ARMS for s in SEEDS]
# -- then re-run the Analysis cells above with DS='lung'

In [12]:
# DS = 'pbmc-immune'
# records = [run_one(DS, s, a) for a in ARMS for s in SEEDS]
# -- then re-run the Analysis cells above with DS='pbmc-immune'

## Export

Writes a markdown block with seed-level `mean +- std` and the effect/noise ratios,
formatted to drop straight into `experiment-results/` or a reviewer reply.

In [13]:
lines = [f'# Seed-variance check -- {DS}', '',
         f'{len(SEEDS)} seeds ({SEEDS}), Stage-1 shared across seeds, '
         f'Stage-2 shortened to train_epochs={COMMON["train_epochs"]}.',
         'Every arm runs at the same reduced setting, so effects and seed spread are',
         'measured on the same footing. Not comparable to the main table by absolute value.', '']

if not df_seed.empty:
    lines += ['## Mean +- std across seeds', '']
    metric_cols = [m for m in present]
    lines.append('| Arm | ' + ' | '.join(metric_cols) + ' |')
    lines.append('|---' * (len(metric_cols) + 1) + '|')
    for arm in df_seed.index:
        cells = []
        for m in metric_cols:
            mu, sd = df_seed.loc[arm, (m, 'mean')], df_seed.loc[arm, (m, 'std')]
            cells.append(f'{mu:.3f}+-{sd:.3f}' if pd.notna(mu) and pd.notna(sd)
                         else (f'{mu:.3f}' if pd.notna(mu) else '-'))
        lines.append(f'| {ARM_DISPLAY.get(arm, arm)} | ' + ' | '.join(cells) + ' |')
    lines.append('')

if not df_eff.empty:
    lines += ['## Effect size vs seed noise', '',
              '| Metric | Arm | full | arm | effect | x seed std |', '|---|---|---|---|---|---|']
    for _, r in df_eff.iterrows():
        ratio = f'{r["ratio_vs_full_std"]:.1f}x' if pd.notna(r['ratio_vs_full_std']) else '-'
        lines.append(f'| {r["metric"]} | {ARM_DISPLAY.get(r["arm"], r["arm"])} | '
                     f'{r["full_mean"]:.3f} | {r["arm_mean"]:.3f} | {r["effect"]:+.3f} | {ratio} |')
    lines.append('')

md = '\n'.join(lines)
with open(os.path.join(OUT_DIR, f'{DS}_seed_variance_summary.md'), 'w') as f:
    f.write(md)
print('saved ->', os.path.join(OUT_DIR, f'{DS}_seed_variance_summary.md'))
print()
print(md)

saved -> /content/drive/MyDrive/rebuttal_results/ablation_seeds/pancreas_seed_variance_summary.md

# Seed-variance check -- pancreas

3 seeds ([31, 1, 2]), Stage-1 shared across seeds, Stage-2 shortened to train_epochs=20.
Every arm runs at the same reduced setting, so effects and seed spread are
measured on the same footing. Not comparable to the main table by absolute value.

## Mean +- std across seeds

| Arm | modularity | purity | batch_entropy | coverage |
|---|---|---|---|---|
| Full model | 0.696+-0.006 | 0.916+-0.006 | 0.419+-0.037 | 0.976+-0.041 |
| - community loss | 0.499+-0.010 | 0.956+-0.001 | 0.478+-0.009 | 0.810+-0.041 |
| - nassoc | 0.698+-0.007 | 0.900+-0.025 | 0.420+-0.071 | 0.929+-0.000 |
| Stop-grad off | 0.626+-0.002 | 0.906+-0.005 | 0.578+-0.022 | 0.929+-0.000 |

## Effect size vs seed noise

| Metric | Arm | full | arm | effect | x seed std |
|---|---|---|---|---|---|
| batch_entropy | Stop-grad off | 0.419 | 0.578 | -0.159 | 4.3x |
| batch_entropy | - community